# MiniGPT from scratch - Colab runner
Runs the whole assignment pipeline end-to-end: tests -> evidence figures -> overfit gate -> training -> evaluation -> attention maps -> ablations.

**Before running:** `Runtime -> Change runtime type -> T4 GPU`, and set `REPO_URL` below to your repository.

In [ ]:
REPO_URL = "https://github.com/<your-username>/transformer-gpt-assignment.git"   # <- EDIT ME
import os
if not os.path.exists("transformer-gpt-assignment"):
    !git clone $REPO_URL transformer-gpt-assignment
%cd transformer-gpt-assignment
!pip install -q pyyaml matplotlib pytest fastapi uvicorn httpx
import torch; print("torch", torch.__version__, "| GPU:", torch.cuda.is_available() and torch.cuda.get_device_name(0))

## 1. Data + unit tests

In [ ]:
!python -m src.dataset --download
!python -m pytest -q

## 2. Evidence figures (hand calculation, mask, shapes, positional encoding, summary)

In [ ]:
!python -m experiments.hand_calculation
!python -m experiments.make_evidence

## 3. Tiny-overfit gate (must collapse to ~0 before any long run)

In [ ]:
!python -m src.train --run-name overfit --overfit-batch --set train.max_steps=300 --set train.batch_size=16 --set train.eval_interval=50

## 4. Train MiniGPT (baseline config, ~minutes on a T4)

In [ ]:
!python -m src.train --config configs/tiny_gpt.yaml --run-name minigpt

## 5. Loss curves, evaluation, generation, attention maps

In [ ]:
!python -m experiments.plot_training --run minigpt
!python -m src.evaluate --checkpoint checkpoints/minigpt.pt
!python -m src.generate --prompt "ROMEO:" --max-new-tokens 200 --temperature 0.8
!python -m experiments.attention_visualization --checkpoint checkpoints/minigpt.pt
!python -m experiments.generation_strategy

## 6. Ablations at full budget (depth / heads / context / position)

In [ ]:
!python -m experiments.ablation --steps 3000 --batch-size 64

## 7. MLM-vs-CLM leakage experiment

In [ ]:
!python -m experiments.mlm_vs_clm

## 8. Look at the figures

In [ ]:
import glob
from IPython.display import Image, display
for f in sorted(glob.glob("outputs/figures/*.png")):
    print(f); display(Image(f))

## 9. Save outputs (checkpoint + figures + logs) to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/minigpt_outputs
!cp -r checkpoints/minigpt.pt outputs /content/drive/MyDrive/minigpt_outputs/
print("saved to MyDrive/minigpt_outputs")